Задание 3

In [33]:
import math

# Линейный штраф

def needleman_wunsch_linear(seq1, seq2, match=1, mismatch=-1, gap=-4):
    n, m = len(seq1), len(seq2)

    F = [[0]*(m+1) for _ in range(n+1)]

    # Инициализация
    for i in range(1, n+1):
        F[i][0] = i * gap
    for j in range(1, m+1):
        F[0][j] = j * gap

    # Заполнение
    for i in range(1, n+1):
        for j in range(1, m+1):
            s = match if seq1[i-1] == seq2[j-1] else mismatch
            F[i][j] = max(
                F[i-1][j-1] + s,
                F[i-1][j] + gap,
                F[i][j-1] + gap
            )

    align1, align2 = "", ""
    i, j = n, m

    while i > 0 or j > 0:
        if i > 0 and j > 0:
            s = match if seq1[i-1] == seq2[j-1] else mismatch
            if F[i][j] == F[i-1][j-1] + s:
                align1 = seq1[i-1] + align1
                align2 = seq2[j-1] + align2
                i -= 1
                j -= 1
                continue

        if i > 0 and F[i][j] == F[i-1][j] + gap:
            align1 = seq1[i-1] + align1
            align2 = "-" + align2
            i -= 1
        else:
            align1 = "-" + align1
            align2 = seq2[j-1] + align2
            j -= 1

    return F, F[n][m], align1, align2


# Аффинный штраф

def needleman_wunsch_affine(seq1, seq2, match=1, mismatch=-1,
                            gap_open=-10, gap_extend=-1):

    n, m = len(seq1), len(seq2)

    M = [[-math.inf]*(m+1) for _ in range(n+1)]
    X = [[-math.inf]*(m+1) for _ in range(n+1)]
    Y = [[-math.inf]*(m+1) for _ in range(n+1)]

    M[0][0] = 0

    # Инициализация
    for i in range(1, n+1):
        X[i][0] = gap_open + (i-1)*gap_extend
    for j in range(1, m+1):
        Y[0][j] = gap_open + (j-1)*gap_extend

    # Заполнение
    for i in range(1, n+1):
        for j in range(1, m+1):
            s = match if seq1[i-1] == seq2[j-1] else mismatch

            M[i][j] = max(
                M[i-1][j-1],
                X[i-1][j-1],
                Y[i-1][j-1]
            ) + s

            X[i][j] = max(
                M[i-1][j] + gap_open,
                X[i-1][j] + gap_extend
            )

            Y[i][j] = max(
                M[i][j-1] + gap_open,
                Y[i][j-1] + gap_extend
            )

    # Определяем финальное состояние
    i, j = n, m
    matrices = {'M': M, 'X': X, 'Y': Y}
    current = max(('M', M[i][j]), ('X', X[i][j]), ('Y', Y[i][j]),
                  key=lambda x: x[1])[0]

    align1, align2 = "", ""

    while i > 0 or j > 0:

        if current == 'M':
            s = match if seq1[i-1] == seq2[j-1] else mismatch
            if M[i][j] == M[i-1][j-1] + s:
                current = 'M'
            elif M[i][j] == X[i-1][j-1] + s:
                current = 'X'
            else:
                current = 'Y'

            align1 = seq1[i-1] + align1
            align2 = seq2[j-1] + align2
            i -= 1
            j -= 1

        elif current == 'X':
            if X[i][j] == M[i-1][j] + gap_open:
                current = 'M'
            else:
                current = 'X'

            align1 = seq1[i-1] + align1
            align2 = "-" + align2
            i -= 1

        elif current == 'Y':
            if Y[i][j] == M[i][j-1] + gap_open:
                current = 'M'
            else:
                current = 'Y'

            align1 = "-" + align1
            align2 = seq2[j-1] + align2
            j -= 1

    score = max(M[n][m], X[n][m], Y[n][m])
    return matrices, score, align1, align2


# Матрицы

def print_matrix(matrix, seq1, seq2, name):
    print(f"\nMatrix {name}:")
    print("      ", "  ".join(["-"] + list(seq2)))
    for i, row in enumerate(matrix):
        if i == 0:
            label = "-"
        else:
            label = seq1[i-1]
        print(label, ["{:4}".format(int(x) if x != -math.inf else -999)
                      for x in row])

# Наш случай

seq1 = "ATGCAGCAGCAGCCA"
seq2 = "ATATAT"

# Линейный
F, score_lin, a1_lin, a2_lin = needleman_wunsch_linear(seq1, seq2)

print("Линейный штраф")
print("Score:", score_lin)
print(a1_lin)
print(a2_lin)
print_matrix(F, seq1, seq2, "F")

# Аффинный
matrices, score_aff, a1_aff, a2_aff = needleman_wunsch_affine(seq1, seq2)

print("Аффинный штраф")
print("Score:", score_aff)
print(a1_aff)
print(a2_aff)

# M — «нормальное чтение текста», X — «мы начали пропускать буквы во второй строке», Y — «мы начали пропускать буквы в первой строке»

print_matrix(matrices['M'], seq1, seq2, "M")
print_matrix(matrices['X'], seq1, seq2, "X")
print_matrix(matrices['Y'], seq1, seq2, "Y")

# Сравнение моделей

print("Сравнение моделей")

print(f"Score (линейный штраф):  {score_lin}")
print(f"Score (аффинный штраф):  {score_aff}")

difference = score_aff - score_lin

print(f"\nРазница (аффинный - линейный): {difference}")

if difference > 0:
    print("Аффинная модель даёт более высокий (лучший) score.")
elif difference < 0:
    print("Линейная модель даёт более высокий (лучший) score.")
else:
    print("Обе модели дают одинаковый score.")

Линейный штраф
Score: -34
ATGCAGCAGCAGCCA
AT-----A-TA---T

Matrix F:
       -  A  T  A  T  A  T
- ['   0', '  -4', '  -8', ' -12', ' -16', ' -20', ' -24']
A ['  -4', '   1', '  -3', '  -7', ' -11', ' -15', ' -19']
T ['  -8', '  -3', '   2', '  -2', '  -6', ' -10', ' -14']
G [' -12', '  -7', '  -2', '   1', '  -3', '  -7', ' -11']
C [' -16', ' -11', '  -6', '  -3', '   0', '  -4', '  -8']
A [' -20', ' -15', ' -10', '  -5', '  -4', '   1', '  -3']
G [' -24', ' -19', ' -14', '  -9', '  -6', '  -3', '   0']
C [' -28', ' -23', ' -18', ' -13', ' -10', '  -7', '  -4']
A [' -32', ' -27', ' -22', ' -17', ' -14', '  -9', '  -8']
G [' -36', ' -31', ' -26', ' -21', ' -18', ' -13', ' -10']
C [' -40', ' -35', ' -30', ' -25', ' -22', ' -17', ' -14']
A [' -44', ' -39', ' -34', ' -29', ' -26', ' -21', ' -18']
G [' -48', ' -43', ' -38', ' -33', ' -30', ' -25', ' -22']
C [' -52', ' -47', ' -42', ' -37', ' -34', ' -29', ' -26']
C [' -56', ' -51', ' -46', ' -41', ' -38', ' -33', ' -30']
A [' -60', ' -55', 

В эволюции и при секвенировании чаще происходят длинные вставки или делеции (например, вставка/выпадение целого повтора, мобильного элемента, ошибки репликации полимеразы и т.д.), чем много независимых одиночных вставок/делеций.
Аффинный штраф сильно поощряет именно такие длинные инделы, а не россыпь одиночных гэпов, что делает выравнивание биологически более правдоподобным.
Именно поэтому в современных программах выравнивания (BLAST, MAFFT, MUSCLE, minimap2 и др.) почти всегда используется аффинная модель штрафов (или даже более сложные её варианты).